Activation de MLFlow

In [19]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Volarus - Retards de vols")

# Active le suivi automatique pour scikit-learn
mlflow.sklearn.autolog()

Importation des données

In [20]:
import pandas as pd

df = pd.read_csv("Envoy_Air_pre_Cleaned.csv")

# Afficher les premières lignes
display(df.head())

# Vérifier le nombre de lignes et de colonnes
print("Dimensions du dataset :", df.shape)

,FlightDate,Origin,Dest,CRSDepTime,ArrDelayMinutes,Distance,Year,Month,DayofMonth,DayOfWeek,...,DestCityMarketID,DestCityName,DestState,DestStateFips,DestStateName,DestWac,DepTimeBlk,CRSArrTime,ArrDel15,ArrTimeBlk
0,2018-10-26,IAH,ORD,1137,0.0,925.0,2018,10,26,5,...,30977,"Chicago, IL",IL,17,Illinois,41,1100-1159,1420,0.0,1400-1459
1,2018-10-27,IAH,ORD,1137,4.0,925.0,2018,10,27,6,...,30977,"Chicago, IL",IL,17,Illinois,41,1100-1159,1420,0.0,1400-1459
2,2018-10-28,IAH,ORD,1137,0.0,925.0,2018,10,28,7,...,30977,"Chicago, IL",IL,17,Illinois,41,1100-1159,1420,0.0,1400-1459
3,2018-10-29,IAH,ORD,1137,0.0,925.0,2018,10,29,1,...,30977,"Chicago, IL",IL,17,Illinois,41,1100-1159,1420,0.0,1400-1459
4,2018-10-30,IAH,ORD,1137,0.0,925.0,2018,10,30,2,...,30977,"Chicago, IL",IL,17,Illinois,41,1100-1159,1420,0.0,1400-1459


Dimensions du dataset : (1033979, 29)


In [21]:
# Nom et type de chaque variable
display(pd.DataFrame({
    "Variable": df.columns,
    "Type": df.dtypes.astype(str).values
}))

,Variable,Type
0,FlightDate,object
1,Origin,object
2,Dest,object
3,CRSDepTime,int64
4,ArrDelayMinutes,float64
5,Distance,float64
6,Year,int64
7,Month,int64
8,DayofMonth,int64
9,DayOfWeek,int64


Variables redondante à retirer

In [22]:
# Suppression des variables redondantes
colonnes_a_supprimer = [
    "OriginCityName",
    "DestCityName",
    "OriginStateName",
    "DestStateName"
]

# Supprimer uniquement les colonnes présentes
colonnes_a_supprimer = [
    col for col in colonnes_a_supprimer if col in df.columns
]

df = df.drop(columns=colonnes_a_supprimer)

# Afficher les résultats
print("Variables supprimées :")
for col in colonnes_a_supprimer:
    print("-", col)

print("\nNombre de variables supprimées :", len(colonnes_a_supprimer))
print("Nombre de variables restantes :", len(df.columns))

Variables supprimées :
- OriginCityName
- DestCityName
- OriginStateName
- DestStateName

Nombre de variables supprimées : 4
Nombre de variables restantes : 25


In [23]:
# Suppression des variables géographiques et horaires redondantes

variables_a_retirer = [
    "OriginAirportID",
    "DestAirportID",
    "OriginStateFips",
    "DestStateFips",
    "OriginCityMarketID",
    "DestCityMarketID",
    "OriginWac",
    "DestWac",
    "DepTimeBlk",
    "ArrTimeBlk"
]

# Supprimer uniquement les colonnes présentes dans df
variables_supprimees = [
    col for col in variables_a_retirer if col in df.columns
]

df = df.drop(columns=variables_supprimees)

# Afficher les résultats
print("Variables supprimées :")
for col in variables_supprimees:
    print("-", col)

print("\nNombre de variables supprimées :", len(variables_supprimees))
print("Nombre de variables restantes :", len(df.columns))

Variables supprimées :
- OriginAirportID
- DestAirportID
- OriginStateFips
- DestStateFips
- OriginCityMarketID
- DestCityMarketID
- OriginWac
- DestWac
- DepTimeBlk
- ArrTimeBlk

Nombre de variables supprimées : 10
Nombre de variables restantes : 15


Préparer les variables explicatives et les cibles

In [24]:
# Variables cibles
y_classification = df["ArrDel15"]
y_regression = df["ArrDelayMinutes"]

# Variables explicatives
X = df.drop(columns=["ArrDel15", "ArrDelayMinutes", "FlightDate"])

# Vérifier les dimensions
print("Variables explicatives (X) :", X.shape)
print("Cible classification :", y_classification.shape)
print("Cible régression :", y_regression.shape)

Variables explicatives (X) : (1033979, 12)
Cible classification : (1033979,)
Cible régression : (1033979,)


Vérification les types de variables

In [25]:
# Afficher le type de chaque variable
display(pd.DataFrame({
    "Variable": X.columns,
    "Type": X.dtypes.astype(str).values
}))

# Compter les variables par type
print("\nNombre de variables numériques :",
      len(X.select_dtypes(include="number").columns))

print("Nombre de variables catégorielles :",
      len(X.select_dtypes(include=["object", "category", "string"]).columns))

,Variable,Type
0,Origin,object
1,Dest,object
2,CRSDepTime,int64
3,Distance,float64
4,Year,int64
5,Month,int64
6,DayofMonth,int64
7,DayOfWeek,int64
8,Flight_Number_Marketing_Airline,int64
9,OriginState,object



Nombre de variables numériques : 8
Nombre de variables catégorielles : 4


Séparer les données pour la classification

In [26]:
# Variables explicatives : sans les deux cibles
X_class = df.drop(
    columns=["ArrDel15", "ArrDelayMinutes", "FlightDate"]
)

# Cible de classification uniquement
y_class = df["ArrDel15"].astype(int)

# Séparation chronologique 80 % / 20 %
limite = int(len(df) * 0.80)

X_class_train = X_class.iloc[:limite].copy()
X_class_test = X_class.iloc[limite:].copy()

y_class_train = y_class.iloc[:limite].copy()
y_class_test = y_class.iloc[limite:].copy()

# Vérifier les dimensions
print("X entraînement :", X_class_train.shape)
print("X test :", X_class_test.shape)
print("y entraînement :", y_class_train.shape)
print("y test :", y_class_test.shape)

X entraînement : (827183, 12)
X test : (206796, 12)
y entraînement : (827183,)
y test : (206796,)


Vérification des variables textuelles

In [27]:
# Identifier les variables catégorielles
variables_textuelles = X_class_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Variables à encoder :")
for col in variables_textuelles:
    print("-", col)

Variables à encoder :
- Origin
- Dest
- OriginState
- DestState


Encoder les variables

In [28]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Identifier les variables catégorielles et numériques
variables_textuelles = X_class_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

variables_numeriques = X_class_train.select_dtypes(
    include=["number"]
).columns.tolist()

# Vérifier le nombre de catégories
print("Nombre de catégories par variable :")

for col in variables_textuelles:
    print(f"{col} : {X_class_train[col].nunique()} catégories")

# Préparer les variables numériques
traitement_numerique = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("standardiser", StandardScaler())
])

# Préparer les variables catégorielles
traitement_categoriel = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combiner les traitements
preprocesseur = ColumnTransformer(transformers=[
    ("numeriques", traitement_numerique, variables_numeriques),
    ("categoriels", traitement_categoriel, variables_textuelles)
])

# Apprendre les transformations sur les données d'entraînement
X_train_encode = preprocesseur.fit_transform(X_class_train)

# Appliquer les mêmes transformations aux données de test
X_test_encode = preprocesseur.transform(X_class_test)

print("\nDimensions après encodage et standardisation :")
print("X_train :", X_train_encode.shape)
print("X_test :", X_test_encode.shape)

Nombre de catégories par variable :
Origin : 187 catégories
Dest : 187 catégories
OriginState : 46 catégories
DestState : 46 catégories


2026/10/28 21:02:57 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '0447130255c944f494dcbd8bd17a9b5a', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/10/28 21:03:06 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\segui\anaconda3\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <http

🏃 View run gentle-flea-112 at: http://localhost:5000/#/experiments/1/runs/0447130255c944f494dcbd8bd17a9b5a
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/10/28 21:03:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/28 21:03:57 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2026/10/28 21:04:21 WARNING mlflow.sklearn: Training metrics will not be recorded because training labels were not specified. To automatically record training metrics, provide training labels as inputs to the model training function.


🏃 View run delicate-donkey-970 at: http://localhost:5000/#/experiments/1/runs/8477d00c2ace4b11b834ce605246ecfd
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/10/28 21:04:25 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\segui\anaconda3\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."



Dimensions après encodage et standardisation :
X_train : (827183, 474)
X_test : (206796, 474)


Créer l'echantillon de 100 000 vols

In [29]:
# Taille de chaque échantillon
taille = 100_000

# Échantillon du début
X_debut = X_train_encode[:taille]
y_debut = y_class_train.iloc[:taille]

# Échantillon de la fin
X_fin = X_train_encode[-taille:]
y_fin = y_class_train.iloc[-taille:]

print("Échantillon du début :", X_debut.shape)
print("Échantillon de la fin :", X_fin.shape)

Échantillon du début : (100000, 474)
Échantillon de la fin : (100000, 474)


In [30]:
# Échantillon du début
limite_debut = int(X_debut.shape[0] * 0.80)

X_train_debut = X_debut[:limite_debut]
X_test_debut = X_debut[limite_debut:]

y_train_debut = y_debut.iloc[:limite_debut]
y_test_debut = y_debut.iloc[limite_debut:]

# Échantillon de la fin
limite_fin = int(X_fin.shape[0] * 0.80)

X_train_fin = X_fin[:limite_fin]
X_test_fin = X_fin[limite_fin:]

y_train_fin = y_fin.iloc[:limite_fin]
y_test_fin = y_fin.iloc[limite_fin:]

print("Début — entraînement :", X_train_debut.shape)
print("Début — test :", X_test_debut.shape)
print("Fin — entraînement :", X_train_fin.shape)
print("Fin — test :", X_test_fin.shape)

Début — entraînement : (80000, 474)
Début — test : (20000, 474)
Fin — entraînement : (80000, 474)
Fin — test : (20000, 474)


Entrainement classification 

In [33]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier


def entrainer_modeles(X_train, y_train, nom_echantillon):
    modeles = {
        "Régression logistique": LogisticRegression(
            max_iter=3000,
            class_weight="balanced"
        ),
        "Arbre de décision": DecisionTreeClassifier(
            max_depth=15,
            class_weight="balanced",
            random_state=42
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=50,
            max_depth=15,
            min_samples_leaf=10,
            class_weight="balanced",
            n_jobs=-1,
            random_state=42
        ),
        "SVM": LinearSVC(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        ),
        "KNN": KNeighborsClassifier(
            n_neighbors=5,
            n_jobs=-1
        )
    }

    for nom, modele in modeles.items():
        print(f"Entraînement : {nom}")

        with mlflow.start_run(run_name=f"{nom_echantillon} - {nom}"):
            mlflow.log_param("echantillon", nom_echantillon)
            mlflow.log_param("modele", nom)

            modele.fit(X_train, y_train)

            mlflow.sklearn.log_model(
                sk_model=modele,
                name="modele",
                skops_trusted_types=[
                    "sklearn.tree._tree.Tree",
                    "scipy.sparse._csr.csr_matrix"
                ]
            )

    return modeles


In [34]:
print("=== ÉCHANTILLON DU DÉBUT ===")
modeles_debut = entrainer_modeles(
    X_train_debut,
    y_train_debut,
    "Début"
)

print("\n=== ÉCHANTILLON DE LA FIN ===")
modeles_fin = entrainer_modeles(
    X_train_fin,
    y_train_fin,
    "Fin"
)

print("\nLes 10 modèles sont entraînés et enregistrés dans MLflow.")

=== ÉCHANTILLON DU DÉBUT ===
Entraînement : Régression logistique


2026/10/28 22:56:49 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2026/10/28 22:56:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Début - Régression logistique at: http://localhost:5000/#/experiments/1/runs/3cd829f32cac47568302aa3021a0a1bd
🧪 View experiment at: http://localhost:5000/#/experiments/1
Entraînement : Arbre de décision


2026/10/28 22:57:24 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2026/10/28 22:57:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Début - Arbre de décision at: http://localhost:5000/#/experiments/1/runs/2ffe516a4449412bae9761b3a23c8bc7
🧪 View experiment at: http://localhost:5000/#/experiments/1
Entraînement : Random Forest


...

🏃 View run Fin - Random Forest at: http://localhost:5000/#/experiments/1/runs/ae75681b993c426dbd728ff1a8359d17
🧪 View experiment at: http://localhost:5000/#/experiments/1
Entraînement : SVM


2026/10/28 23:42:49 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2026/10/28 23:42:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Fin - SVM at: http://localhost:5000/#/experiments/1/runs/77c54295de0b4b2b9887826a74579187
🧪 View experiment at: http://localhost:5000/#/experiments/1
Entraînement : KNN


2026/10/28 23:43:16 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2026/10/28 23:43:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Fin - KNN at: http://localhost:5000/#/experiments/1/runs/57dc9f125b5a478eb6a1628f6d5e38fa
🧪 View experiment at: http://localhost:5000/#/experiments/1

Les 10 modèles sont entraînés et enregistrés dans MLflow.


Comparer les modèles de classification

In [35]:

import pandas as pd
import mlflow
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

resultats = []

# Évaluer les modèles du début
for nom, modele in modeles_debut.items():
    print(f"Évaluation début : {nom}...")

    predictions = modele.predict(X_test_debut)

    metrics = {
        "Accuracy": accuracy_score(y_test_debut, predictions),
        "Precision": precision_score(
            y_test_debut, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test_debut, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_test_debut, predictions, zero_division=0
        )
    }

    with mlflow.start_run(
        run_name=f"Évaluation Début - {nom}"
    ):
        mlflow.log_param("echantillon", "Début")
        mlflow.log_param("modele", nom)
        mlflow.log_metrics(metrics)

    resultats.append({
        "Période": "Début",
        "Modèle": nom,
        **metrics
    })


# Évaluer les modèles de la fin
for nom, modele in modeles_fin.items():
    print(f"Évaluation fin : {nom}...")

    predictions = modele.predict(X_test_fin)

    metrics = {
        "Accuracy": accuracy_score(y_test_fin, predictions),
        "Precision": precision_score(
            y_test_fin, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test_fin, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_test_fin, predictions, zero_division=0
        )
    }

    with mlflow.start_run(
        run_name=f"Évaluation Fin - {nom}"
    ):
        mlflow.log_param("echantillon", "Fin")
        mlflow.log_param("modele", nom)
        mlflow.log_metrics(metrics)

    resultats.append({
        "Période": "Fin",
        "Modèle": nom,
        **metrics
    })


df_resultats = pd.DataFrame(resultats)

display(
    df_resultats.sort_values(
        by=["Période", "F1-score"],
        ascending=[True, False]
    ).round(4)
)


Évaluation début : Régression logistique...
🏃 View run Évaluation Début - Régression logistique at: http://localhost:5000/#/experiments/1/runs/a25330c7301246108a7636437a508a7a
🧪 View experiment at: http://localhost:5000/#/experiments/1
Évaluation début : Arbre de décision...
🏃 View run Évaluation Début - Arbre de décision at: http://localhost:5000/#/experiments/1/runs/0a23c18cab8847e4800c706b62394fc4
🧪 View experiment at: http://localhost:5000/#/experiments/1
Évaluation début : Random Forest...
🏃 View run Évaluation Début - Random Forest at: http://localhost:5000/#/experiments/1/runs/6595e08427e84de7ba838c62b334e862
🧪 View experiment at: http://localhost:5000/#/experiments/1
Évaluation début : SVM...
🏃 View run Évaluation Début - SVM at: http://localhost:5000/#/experiments/1/runs/d4f42e9c7224424b95af13e9ba52e83f
🧪 View experiment at: http://localhost:5000/#/experiments/1
Évaluation début : KNN...
🏃 View run Évaluation Début - KNN at: http://localhost:5000/#/experiments/1/runs/7286413f3

,Période,Modèle,Accuracy,Precision,Recall,F1-score
1,Début,Arbre de décision,0.6517,0.3290,0.5773,0.4191
2,Début,Random Forest,0.6158,0.3009,0.5785,0.3959
3,Début,SVM,0.6272,0.2879,0.4838,0.3610
0,Début,Régression logistique,0.6315,0.2889,0.4744,0.3591
4,Début,KNN,0.7570,0.3045,0.0905,0.1395
7,Fin,Random Forest,0.5747,0.1930,0.5358,0.2838
8,Fin,SVM,0.5563,0.1851,0.5355,0.2751
5,Fin,Régression logistique,0.5574,0.1852,0.5339,0.2750
6,Fin,Arbre de décision,0.6029,0.1810,0.4328,0.2553
9,Fin,KNN,0.8021,0.1978,0.0846,0.1185


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Probabilités sur le jeu de test normal
probabilites = modele_logistique.predict_proba(X_test_encode)[:, 1]

# Tester plusieurs seuils
seuils = [0.50, 0.40, 0.30, 0.20]

resultats_seuils = []

for seuil in seuils:
    predictions = (probabilites >= seuil).astype(int)

    resultats_seuils.append({
        "Seuil": seuil,
        "Precision": precision_score(
            y_class_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_class_test, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_class_test, predictions, zero_division=0
        )
    })

df_seuils = pd.DataFrame(resultats_seuils)

display(df_seuils.round(4))

,Seuil,Precision,Recall,F1-score
0,0.5,0.2202,0.4078,0.2860
1,0.4,0.1967,0.6840,0.3056
2,0.3,0.1802,0.9070,0.3007
3,0.2,0.1732,0.9904,0.2949
